[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/adv-14-clip-in-miniature.ipynb)

# Advanced Discussion 14 — CLIP in miniature

A symmetric contrastive image-text model on digits with attribute captions, zero-shot classification, held-out classes, and the effect of temperature.

**Optional advanced-discussion lab** for Generative AI and Large Language Models with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier; CPU is enough unless a note says otherwise.

Uses scikit-learn's bundled digits; runs in about a minute.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
torch.manual_seed(0); np.random.seed(0)

## 1. a toy image-text dataset: digit images, and captions built from words about them

In [ ]:
X, y = load_digits(return_X_y=True); X = X / 16.0
WORDS = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine"]
ATTR = {"even": lambda d: d % 2 == 0, "odd": lambda d: d % 2 == 1, "small": lambda d: d < 5, "large": lambda d: d >= 5, "round": lambda d: d in (0, 6, 8, 9), "straight": lambda d: d in (1, 4, 7)}
vocab = WORDS + list(ATTR); tok = {w: i for i, w in enumerate(vocab)}
def caption(d, rng, use_name=True):
    words = [w for w, f in ATTR.items() if f(d)]                       # attribute words that are true for the digit
    rng.shuffle(words)
    ws = words[:rng.randint(2, len(words) + 1)]
    if use_name: ws = ws + [WORDS[d]]
    return ws
HELD = [7, 8]                                                          # classes never seen with their NAME during training
rng = np.random.RandomState(0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
T = lambda a: torch.tensor(a, dtype=torch.float32)

class TextEnc(nn.Module):                                              # bag of word embeddings -> MLP
    def __init__(s, d=32): super().__init__(); s.emb = nn.EmbeddingBag(len(vocab), 64, mode="mean"); s.mlp = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, d))
    def forward(s, batch_words):
        idx = torch.cat([torch.tensor([tok[w] for w in ws]) for ws in batch_words]); off = torch.tensor([0] + [len(ws) for ws in batch_words[:-1]]).cumsum(0)
        return s.mlp(s.emb(idx, off))
img_enc = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 32)); txt_enc = TextEnc()
log_scale = nn.Parameter(torch.tensor(2.0))                            # learned temperature (logit scale), as in CLIP
opt = torch.optim.Adam(list(img_enc.parameters()) + list(txt_enc.parameters()) + [log_scale], lr=3e-3)
def clip_loss(zi, zt):
    zi, zt = F.normalize(zi, dim=1), F.normalize(zt, dim=1); logits = log_scale.exp() * zi @ zt.T
    tgt = torch.arange(len(zi)); return (F.cross_entropy(logits, tgt) + F.cross_entropy(logits.T, tgt)) / 2

## 2. train with the symmetric InfoNCE loss; held-out digits keep image + attribute words but never their name

In [ ]:
for epoch in range(80):
    perm = rng.permutation(len(Xtr))
    for i in range(0, len(perm), 128):
        b = perm[i:i + 128]
        if len(b) < 8: continue
        caps = [caption(ytr[j], rng, use_name=ytr[j] not in HELD) for j in b]
        loss = clip_loss(img_enc(T(Xtr[b])), txt_enc(caps)); opt.zero_grad(); loss.backward(); opt.step()
print("final training loss %.3f, learned logit scale %.1f" % (loss.item(), log_scale.exp().item()))

## 3. evaluation: zero-shot classification by comparing an image with a prompt for each class

In [ ]:
@torch.no_grad()
def zero_shot(prompts, classes):
    zt = F.normalize(txt_enc(prompts), dim=1); zi = F.normalize(img_enc(T(Xte)), dim=1); pred = (zi @ zt.T).argmax(1).numpy()
    m = np.isin(yte, classes); return (np.array(classes)[pred[m]] == yte[m]).mean()
name_prompts = [[WORDS[d]] for d in range(10)]                         # "seven"
attr_prompts = [[w for w, f in ATTR.items() if f(d)] for d in range(10)]   # "odd large straight"
seen = [d for d in range(10) if d not in HELD]
print("zero-shot accuracy with NAME prompts on the 8 seen classes (chance 12.5%%): %.3f" % zero_shot([name_prompts[d] for d in seen], seen))
# held-out classes can only be reached through their attribute words
print("zero-shot accuracy with ATTRIBUTE prompts over all 10 classes (chance 10%%): %.3f" % zero_shot(attr_prompts, list(range(10))))
with torch.no_grad():
    zt = F.normalize(txt_enc(attr_prompts), dim=1); zi = F.normalize(img_enc(T(Xte)), dim=1); pred = (zi @ zt.T).argmax(1).numpy()
for h in HELD: print("  held-out digit %d (never trained with its name): recognised via attribute prompt %.3f" % (h, (pred[yte == h] == h).mean()))
print("  overall accuracy of attribute prompts on the 8 seen classes %.3f" % np.mean([(pred[yte == d] == d).mean() for d in seen]))
# prompts a supervised classifier cannot answer: attribute combos never used as a class label
with torch.no_grad():
    q = txt_enc([["odd", "large", "straight"], ["even", "small", "straight"]]); qs = F.normalize(q, dim=1) @ zi.T
    for name, row in zip(("odd large straight", "even small straight"), qs):
        top = row.topk(30).indices.numpy(); print("  query '%s': digits among top-30 images: %s" % (name, dict(zip(*np.unique(yte[top], return_counts=True)))))
# retrieval
with torch.no_grad():
    caps = [caption(d, np.random.RandomState(d), use_name=True) for d in range(10)]; zt = F.normalize(txt_enc(caps), dim=1); sim = zi @ zt.T
print("image -> caption retrieval among 10 captions, top-1 accuracy: %.3f" % (sim.argmax(1).numpy() == yte).mean())

## 4. what temperature does: same embeddings, different logit scale

In [ ]:
zi_, zt_ = F.normalize(img_enc(T(Xtr[:128])), dim=1).detach(), F.normalize(txt_enc([caption(d, rng) for d in ytr[:128]]), dim=1).detach()
for s in (1, 10, 30, 100):
    logits = s * zi_ @ zt_.T; print("  logit scale %3d -> contrastive loss %.3f" % (s, F.cross_entropy(logits, torch.arange(128)).item()))

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.